# KneeVision++ — X-ray Training

Run the first cell, then follow the instructions for your environment.

In [ ]:
# --- Colab Setup ---
import sys, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    # Upload src/ and data/ to Colab runtime or symlink from Drive
    # Option A: Use your Google Drive
    project_root = Path("/content/drive/MyDrive/KneeVision")
    # Option B: Upload via Colab file panel (then set path manually)
    # project_root = Path("/content/KneeVision")

    !pip install torch torchvision numpy pillow tqdm -q
else:
    project_root = Path.cwd().parent

sys.path.insert(0, str(project_root / "src"))
os.chdir(project_root)
print(f"Project root: {project_root}")
print(f"Working dir:  {os.getcwd()}")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from kneevision.config.settings import RAW_DATA_DIR, BATCH_SIZE, LEARNING_RATE, NUM_EPOCHS
from kneevision.models.image_model import KneeXRayClassifier
from kneevision.data.dataset import KneeXRayDataset
from kneevision.data.transforms import train_transform, val_transform
from kneevision.training.trainer import train_epoch, validate
from kneevision.utils.helpers import set_seed, get_device

set_seed(42)
device = get_device()
print(f"Device: {device}")
print(f"Data path: {RAW_DATA_DIR} (exists: {RAW_DATA_DIR.exists()})")

In [ ]:
def get_paths_and_labels(split: str):
    paths, labels = [], []
    split_dir = RAW_DATA_DIR / split
    if not split_dir.exists():
        raise FileNotFoundError(f"Data not found at {split_dir}. Upload the dataset or check the path.")
    for grade_dir in sorted(split_dir.iterdir()):
        if not grade_dir.is_dir():
            continue
        label = int(grade_dir.name)
        for img_path in sorted(grade_dir.glob("*.png")):
            paths.append(img_path)
            labels.append(label)
    return paths, labels

train_paths, train_labels = get_paths_and_labels("train")
val_paths, val_labels = get_paths_and_labels("val")
print(f"Train: {len(train_paths)} images")
print(f"Val:   {len(val_paths)} images")

In [ ]:
train_ds = KneeXRayDataset(train_paths, train_labels, train_transform)
val_ds = KneeXRayDataset(val_paths, val_labels, val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
model = KneeXRayClassifier("densenet121", 5).to(device)

class_counts = torch.tensor([2286, 1046, 1516, 757, 173], dtype=torch.float)
class_weights = (1.0 / class_counts) * class_counts.sum() / 5
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

save_dir = Path("models")
save_dir.mkdir(exist_ok=True)

In [ ]:
best_acc = 0.0
train_losses, val_losses, val_accs = [], [], []

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    print(f"Epoch {epoch:2d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), save_dir / "best_model.pt")
        print(f"  -> Saved best model (acc={val_acc:.4f})")

print(f"\nBest val accuracy: {best_acc:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, label="Train Loss")
ax1.plot(val_losses, label="Val Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True)

ax2.plot(val_accs, label="Val Accuracy", color="green")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()